# Fake Profile Detection on Social Media using Machine Learning

## 1. Problem Statement
The goal of this project is to classify social media accounts as **REAL** or **FAKE** based on profile and behavioral features. This is a critical task for maintaining the integrity of social platforms and preventing misinformation.

## 2. Dataset Overview
We use a dataset containing features for both real and fake Instagram accounts. Key features include:
- Profile picture presence
- Username numerical ratio
- Full name length
- Description length
- Number of posts, followers, and following count
- Account privacy status

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Load data
train_df = pd.read_csv('../data/social_media_train.csv')
test_df = pd.read_csv('../data/social_media_test.csv')

# Initial look
print(f"Training data shape: {train_df.shape}")
train_df.head()

## 3. Exploratory Data Analysis (EDA)
EDA helps us understand patterns in the data. For example, fake accounts often have fewer posts and a higher ratio of following to followers.

In [ ]:
# Distribution of Target Variable
sns.countplot(x='fake', data=train_df)
plt.title('Distribution of Fake vs Real Accounts')
plt.show()

## 4. Feature Engineering
We convert categorical features like 'Yes/No' to numerical values (1/0) and create a new feature: the ratio of followers to following.

In [ ]:
def preprocess(df):
    df = df.copy()
    # Encode categoricals
    df['profile_pic'] = df['profile_pic'].map({'Yes': 1, 'No': 0})
    df['extern_url'] = df['extern_url'].map({'Yes': 1, 'No': 0})
    df['private'] = df['private'].map({'Yes': 1, 'No': 0})
    df['sim_name_username'] = df['sim_name_username'].map({'No match': 0, 'Partial match': 1, 'Full match': 2})
    
    # New feature: follower to following ratio
    df['follower_following_ratio'] = df['num_followers'] / (df['num_following'] + 1)
    
    # Drop any leftover index column
    if df.columns[0] == 'Unnamed: 0' or df.columns[0] == '':
        df = df.iloc[:, 1:]
        
    return df

train_processed = preprocess(train_df)
test_processed = preprocess(test_df)

## 5. Model Building & Training
We compare three models:
1. **Logistic Regression**: A simple yet effective baseline.
2. **Random Forest**: An ensemble method that uses multiple decision trees to improve accuracy and reduce overfitting.
3. **Support Vector Machine (SVM)**: Finds the optimal hyperplane to separate classes.

In [ ]:
X_train = train_processed.drop('fake', axis=1)
y_train = train_processed['fake']
X_test = test_processed.drop('fake', axis=1)
y_test = test_processed['fake']

# Scaling numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize models
lr = LogisticRegression()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(probability=True)

# Train
lr.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)
svm.fit(X_train_scaled, y_train)

## 6. Evaluation
We evaluate the models using Accuracy, Precision, and **Recall**. 

**Why is Recall important for detecting fake profiles?**
Recall (or Sensitivity) measures how many of the actual fake profiles were correctly identified. In social media moderation, it's often better to catch as many fake profiles as possible (high recall), even if some real ones are flagged for review, to prevent large-scale spam or bot attacks.

In [ ]:
models = {'Logistic Regression': lr, 'Random Forest': rf, 'SVM': svm}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))
    print("\n")

## 7. Feature Importance
Using Random Forest, we can see which features contribute most to the classification.

In [ ]:
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_df)
plt.title('Feature Importance (Random Forest)')
plt.show()